# XCoder-0.8B - Fine-tune TEMPATAN (GTX 1070 8GB)

Tanpa Colab/Kaggle: guna `finetune/train_local_qwen35.py` (transformers + peft + trl,
fp32 - Pascal tiada Unsloth) pada PC ini.

**Panduan**
1. Data: `datasets/all.jsonl` (jana di terminal: `python datasets/scripts/merge_one.py --force`;
   `fable_agent_ms.jsonl` digabung automatik jika wujud). Fail ini `git-ignored` - kekal tempatan.
2. Kernel notebook: mana-mana pun boleh - sel di bawah memanggil `.venv-gpu\Scripts\python.exe` terus.
3. Jalankan sel dari atas ke bawah. Jangka masa berjam-jam (GTX 1070 jauh lebih perlahan dari T4) -
   biarkan semalaman. Checkpoint setiap 100 langkah membolehkan henti/sambung.
4. Terputus / OOM / PC dimatikan? Run sel training semula - auto-resume dari
   `finetune/runs/xcoder-best-0.8b/`. OOM? Tutup app GPU lain atau tambah `--max-len 2048`.
5. Selepas training, sel GGUF tukar merged 16-bit -> `xcoder-best-0.8b-q4_k_m.gguf` (daftar di sel terakhir).

Pilihan: `--name xcoder-xxx` tukar nama output, `--out DIR` tukar folder checkpoint,
`--data docker,linux` latih stack tertentu (bukan keseluruhan `all`).

In [ ]:
!nvidia-smi

In [ ]:
import os
from pathlib import Path

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "finetune" / "train_local_qwen35.py").exists()), None)
if ROOT is None:
    raise SystemExit("buka notebook ini dari dalam repo trainmyai")
os.chdir(ROOT)

VENV = ROOT / ".venv-gpu" / "Scripts" / "python.exe"
DATA = ROOT / "datasets" / "all.jsonl"
if not VENV.exists():
    raise SystemExit("tiada .venv-gpu - ikut arahan setup di atas finetune/train_local_qwen35.py")
if not DATA.exists():
    raise SystemExit("datasets/all.jsonl tiada - jalankan: python datasets/scripts/merge_one.py --force")

rows = sum(1 for line in DATA.open(encoding="utf-8") if line.strip())
print("repo :", ROOT)
print("data :", DATA, f"{DATA.stat().st_size / 1e6:.1f} MB, {rows} rows")

In [ ]:
# Semak bilangan baris + status checkpoint (tanpa mula training)
! "{VENV}" finetune\train_local_qwen35.py --data all --dry

In [ ]:
# LATIHAN PENUH - berjam-jam. Boleh berhenti bila-bila masa (Kernel > Interrupt / tutup PC):
# run semula sel ini akan sambung dari checkpoint terakhir (auto-resume).
# OOM 8GB VRAM? tutup app GPU lain, atau tambah --max-len 2048 pada arahan di bawah.
! "{VENV}" finetune\train_local_qwen35.py --data all

In [ ]:
# Jalankan SELEPAS training selesai (print "exported ..." oleh sel sebelumnya).
# merged 16-bit -> f16 GGUF -> q4_k_m (~500 MB, sedia untuk /model add). Ambil ~5-15 min.
import os

MERGED = "finetune/runs/xcoder-best-0.8b-merged"
GGUF = "xcoder-best-0.8b-q4_k_m.gguf"
if not os.path.isdir(MERGED):
    print("merged tiada - selesaikan sel training dahulu")
else:
    ! "{VENV}" build\train\llama.cpp\convert_hf_to_gguf.py {MERGED} --outfile tmp-f16.gguf --outtype f16
    ! "{ROOT}\app\bin\llama-quantize.exe" tmp-f16.gguf {GGUF} Q4_K_M
    if os.path.exists("tmp-f16.gguf"):
        os.remove("tmp-f16.gguf")
    if os.path.exists(GGUF):
        print("sedia:", os.path.abspath(GGUF), round(os.path.getsize(GGUF) / 1e6), "MB")

## Selepas ini (di PC)
`xcoder-best-0.8b-q4_k_m.gguf` kini di root repo.

1. Daftar + aktifkan (dalam CLI): `/model add xcoder-best-0.8b-q4_k_m.gguf --name xcoder-best --domains general`
2. Uji identiti: tanya "Siapa anda?" / "Are you Qwen?" -> "Saya XCoder..." tanpa sebut Qwen/Alibaba.
3. (Pilihan) Held-out loss: `.venv-gpu\Scripts\python.exe finetune\eval_loss.py --model finetune\runs\xcoder-best-0.8b-merged --eval finetune\runs\xcoder-best-0.8b\eval.jsonl`